# Synthetic two-speaker conversations for caramel

Text from an OpenAI-compatible teacher (Ollama here) or a local MLX model, spoken by two voices (Phonon or Qwen3-TTS
cloning), placed on one timeline with barge-ins, hold scenes (*wait*, *listen*) and backchannels.

caramel never sees text, so the text and word timestamps only drive the timing. What it trains on is the two tracks:
each episode's `assistant.wav` and `user.wav` become speakers A and B of a `prepare.py --manifest` row, and train.py
uses every conversation from both sides.

Install the extras once: `uv pip install openai datasets ptts mlx-vlm tqdm`.

### Resume an interrupted run
Keep `OUTPUT_DIR` and the settings unchanged and rerun. Saved episodes and speech caches are reused. Timed-out
teacher requests go to `teacher_failures.jsonl`; retry them with `generator.generate(..., retry_failed=True)`.

In [ ]:
import sys; sys.path.insert(0, "..")
from pathlib import Path
from openai import OpenAI
from synthetic.generator import SyntheticConfig, SyntheticConversationGenerator
from synthetic.phonon_speech import PhononSpeech, PhononVoice, phonon_voices
from synthetic.qwen_speech import Qwen3ClonedSpeech, VoiceReference
from synthetic.teacher import ChatTeacher, MLXTeacher, load_prompt_corpus
from synthetic.caramel_export import write_caramel_manifests

In [ ]:
OUTPUT_DIR = Path("../data/synthetic")
MAX_SAMPLES = 500 # None = whole prompt corpus
SYSTEM_PROMPT = "You are JOSIE, a voice assistant. Your replies are spoken aloud: no markdown, code or math."

TEACHER = "ollama"   # "mlx": local MLX-VLM model, "ollama": OpenAI-compatible endpoint
TTS = "phonon"    # "phonon": Gradium Phonon (multilingual, on-device) | "qwen": Qwen3-TTS voice cloning

if TEACHER == "mlx":
    teacher = MLXTeacher("mlx-community/gemma-4-e4b-it-4bit", max_tokens=512)
elif TEACHER == "ollama":
    # Fail visibly after a stalled request instead of waiting through SDK retries.
    client = OpenAI(
        base_url="http://localhost:11434/v1", api_key="ollama",
        timeout=120.0, max_retries=0,
    )
    teacher = ChatTeacher(client, assistant_model="mistral-large-4:cloud", followup_model="deepseek-v4.1-flash:cloud")
else:
    raise ValueError(f"Unknown TEACHER: {TEACHER}")

if TTS == "phonon":
    # phonon-40m-multilingual is gated (CC BY-NC 4.0): accept the terms on Hugging Face, then `hf auth login`.
    # Voices: PhononVoice(name="Zoey") from `python -m ptts --list-voices`, PhononVoice(file="voice.safetensors"),
    # or PhononVoice(wav="reference.wav") if the checkpoint supports cloning.
    # User voice: one is picked at random per conversation from this pool. Select by name or filter, e.g.
    #   phonon_voices("Zoey", "Hugh", "Maeve")  |  phonon_voices(accent="uk")  |  phonon_voices(gender="male")
    ASSISTANT_VOICE = "Freya"
    USER_VOICES = phonon_voices(exclude=[ASSISTANT_VOICE])   # all 25 US + UK catalog voices except JOSIE's
    speech = PhononSpeech(
        OUTPUT_DIR / "_speech_cache",
        model="hf://gradium/phonon-40m-multilingual",   # or "hf://gradium/phonon-31m-en" (Apache 2.0)
        language="en",
        user_voice=USER_VOICES,
        assistant_voice=PhononVoice(name=ASSISTANT_VOICE),
        min_alignment_match=0.8,
    )
else:
    speech = Qwen3ClonedSpeech(
        OUTPUT_DIR / "_speech_cache",
        model="mlx-community/Qwen3-TTS-12Hz-0.6B-Base-8bit",
        alignment_model="mlx-community/Qwen3-ForcedAligner-0.6B-8bit",
        alignment_language="English",
        language="english",
        user_reference=VoiceReference(Path("../data/voice_references/your_voice.wav"),
                                      "Hello, this is my natural speaking voice. I am recording a short sample for JOSIE."),
        assistant_reference=VoiceReference(Path("../data/voice_references/josie_voice.wav"),
                                           "Good morning. The weather is clear, and the temperature is 23 degrees."),
        min_alignment_match=0.8,
    )
config = SyntheticConfig(
    max_turns=8, barge_in_probability=0.2, assistant_stop_ms=750, system_prompt=SYSTEM_PROMPT, seed=42,
    wait_probability=0.15,         # "can you wait a minute?" -> silent while the user is busy, then resume
    listen_probability=0.10,       # "listen until I finish my presentation" -> silent, then feedback
    max_wait_seconds=60,           # longest timed wait ("a minute")
    check_in_probability=0.5,      # "the minute is up, can I continue?" / "not yet, I'll tell you"
    backchannel_probability=0.5,   # "mhm", "right", "yeah" in the pauses of long user turns
)


In [ ]:
prompts, source = load_prompt_corpus("mlx-community/ultrafeedback-prompts-flat-rlhf")
generator = SyntheticConversationGenerator(teacher, speech, OUTPUT_DIR, config, source_metadata=source)
# Capture all thread stacks if TTS, alignment, or the teacher stops progressing.
# The log also works in kernels whose stderr does not expose a file descriptor.
import faulthandler
stall_log = OUTPUT_DIR / "generation_stall.log"
print(f"Stall diagnostics (after 120 seconds): {stall_log.resolve()}", flush=True)
with stall_log.open("w") as diagnostic_file:
    faulthandler.dump_traceback_later(120, repeat=True, file=diagnostic_file)
    try:
        episodes = generator.generate(prompts, max_samples=MAX_SAMPLES)
    finally:
        faulthandler.cancel_dump_traceback_later()
print(f"{len(episodes)} conversations, {sum(len(e['barge_ins']) for e in episodes)} barge-ins")

## Tokenize for caramel
Writes `caramel_train.jsonl` / `caramel_validation.jsonl` (complete episodes of at least 10 s), then encodes them with
Mimi into token stores for `train.py`. Stores are appended to, so rerunning after more generation adds only what you
pass in — point `--out` at a fresh folder or delete the old one first to avoid duplicates.

In [ ]:
counts = write_caramel_manifests(OUTPUT_DIR, min_seconds=10.0)
print(counts)

In [ ]:
import subprocess
for split in ("train", "validation"):
    if counts[split]:
        subprocess.run([sys.executable, "prepare.py", str((OUTPUT_DIR / f"caramel_{split}.jsonl").resolve()),
                        "--manifest", "--out", str((OUTPUT_DIR.parent / f"synthetic_tokens_{split}").resolve())],
                       cwd="..", check=True)